# Qwen2.5-VL Soft Prompt Tuning with PEFT on RSICD

This notebook fine-tunes a soft prompt for Qwen2.5-VL 7B on the RSICD dataset using PEFT.

by [Gayanuka Amarasuriya](https://gayanukaa.github.io/)

In [ ]:
!pip install -q torch transformers peft datasets evaluate accelerate

In [16]:
import torch
from datasets import load_dataset
from transformers import AutoProcessor, AutoModel, TrainingArguments, Trainer
from peft import PromptTuningConfig, get_peft_model, TaskType
import evaluate
import time
from PIL import Image

## Configurations


In [2]:
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
DATASET_NAME = "arampacha/rsicd"
MAX_LENGTH = 128
PROMPT_LENGTH = 4
BATCH_SIZE = 1
NUM_TRAIN_EPOCHS = 3
OUTPUT_DIR = "./qwen_rsicd_output"
device = "cuda" if torch.cuda.is_available() else "cpu"

## Load Dataset


In [11]:
dataset = load_dataset(DATASET_NAME)

dataset

DatasetDict({
    train: Dataset({
        features: ['filename', 'captions', 'image'],
        num_rows: 8734
    })
    test: Dataset({
        features: ['filename', 'captions', 'image'],
        num_rows: 1093
    })
    valid: Dataset({
        features: ['filename', 'captions', 'image'],
        num_rows: 1094
    })
})

In [12]:
# Use existing splits from the dataset - take subset for faster training
train_dataset = dataset["train"].select(range(1000))  # First 500 samples for training
eval_dataset = dataset["valid"].select(range(200))   # First 100 samples for evaluation

In [13]:
print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")
print(f"Features: {train_dataset.features}")

Training samples: 1000
Evaluation samples: 200
Features: {'filename': Value(dtype='string', id=None), 'captions': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'image': Image(mode=None, decode=True, id=None)}


## Load Model and Processor


In [17]:
processor = AutoProcessor.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

## Apply Prompt Tuning using PEFT


In [18]:
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=PROMPT_LENGTH,
    tokenizer_name_or_path=MODEL_NAME,
)

model = get_peft_model(model, peft_config)

AttributeError: 'Qwen2_5_VLModel' object has no attribute 'prepare_inputs_for_generation'

## Preprocess Dataset


In [ ]:
def preprocess(example):
    image = example["image"]

    # Format as conversation for Qwen2.5-VL
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Describe this satellite image."}
            ]
        }
    ]

    # Apply chat template
    input_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Process inputs
    inputs = processor(
        text=input_text,
        images=image,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

    # Create labels from the first caption
    target_text = example["captions"][0]
    labels = processor.tokenizer(
        target_text,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).input_ids

    # Prepare output
    result = {
        "input_ids": inputs["input_ids"].squeeze(),
        "attention_mask": inputs["attention_mask"].squeeze(),
        "labels": labels.squeeze()
    }

    # Add pixel_values if present
    if "pixel_values" in inputs:
        result["pixel_values"] = inputs["pixel_values"].squeeze()

    return result

train_dataset = train_dataset.map(preprocess)
eval_dataset = eval_dataset.map(preprocess)
train_dataset.set_format(type="torch")
eval_dataset.set_format(type="torch")

## Training Arguments


In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    logging_steps=10,
    save_steps=100,
    save_total_limit=1,
    fp16=True,
    report_to="none",
    remove_unused_columns=False,  # Important for vision models
    dataloader_pin_memory=False,  # Can help with memory issues
    gradient_checkpointing=True   # Save memory during training
)

## Evaluation Metrics


In [ ]:
cider = evaluate.load("cider")
spice = evaluate.load("spice")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = torch.argmax(torch.tensor(logits), dim=-1)

    # Decode predictions and labels
    decoded_preds = processor.tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Clean up the predictions (remove input prompt)
    cleaned_preds = []
    for pred in decoded_preds:
        # Extract only the generated part after the prompt
        if "assistant" in pred:
            cleaned_pred = pred.split("assistant")[-1].strip()
        else:
            cleaned_pred = pred.strip()
        cleaned_preds.append(cleaned_pred)

    try:
        cider_score = cider.compute(predictions=cleaned_preds, references=[[l] for l in decoded_labels])["score"]
    except:
        cider_score = 0.0

    try:
        spice_score = spice.compute(predictions=cleaned_preds, references=[[l] for l in decoded_labels])["score"]
    except:
        spice_score = 0.0

    return {
        "CIDEr": cider_score,
        "SPICE": spice_score
    }

## Train the Model


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=processor.tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

## Inference Time and VRAM Usage


## Track Metrics on Multiple Inference Samples


In [ ]:
# Test on multiple samples and track metrics
num_samples = 10
predictions = []
references = []

for i in range(min(num_samples, len(eval_dataset))):
    sample = eval_dataset[i]

    # Prepare inference input
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text", "text": "Describe this satellite image."}
            ]
        }
    ]

    input_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=input_text, images=sample["image"], return_tensors="pt").to(device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.7,
            do_sample=True
        )

    # Decode output
    generated_text = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract generated part
    if "assistant" in generated_text:
        generated_part = generated_text.split("assistant")[-1].strip()
    else:
        generated_part = generated_text.strip()

    predictions.append(generated_part)
    references.append([sample["labels"]])  # Assuming labels contain the reference caption

    print(f"Sample {i+1}:")
    print(f"Generated: {generated_part}")
    print(f"Reference: {sample['labels']}")
    print("-" * 50)

# Calculate metrics
try:
    cider_score = cider.compute(predictions=predictions, references=references)["score"]
    print(f"CIDEr Score: {cider_score:.4f}")
except Exception as e:
    print(f"CIDEr calculation failed: {e}")

try:
    spice_score = spice.compute(predictions=predictions, references=references)["score"]
    print(f"SPICE Score: {spice_score:.4f}")
except Exception as e:
    print(f"SPICE calculation failed: {e}")

In [ ]:
torch.cuda.reset_peak_memory_stats()
start = time.time()

# Get a sample from the original dataset
sample = eval_dataset[0]

# Prepare inference input
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": sample["image"]},
            {"type": "text", "text": "Describe this satellite image."}
        ]
    }
]

input_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=input_text, images=sample["image"], return_tensors="pt").to(device)

# Generate
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.7,
        do_sample=True
    )

end = time.time()

# Decode output
generated_text = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated:", generated_text)
print("Inference Time:", end - start, "seconds")
print("Peak VRAM:", torch.cuda.max_memory_allocated() / 1e9, "GB")